In [ ]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset


def read_points(file_path):
    suffix = os.path.splitext(file_path)[1]
    if suffix != '.bin':
        raise ValueError("只支援 .bin 檔")

    raw = np.fromfile(file_path, dtype=np.float32)

    if raw.size % 7 == 0:
        dim = 7
    elif raw.size % 4 == 0:
        dim = 4
    else:
        raise ValueError(f"File size({raw.size}) cannot be divided by 4 or 7 floats.")

    data = raw.reshape(-1, dim)
    return data[:, :4] if dim == 7 else data


def bbox_camera2lidar(bboxes, tr_velo_to_cam, r0_rect):
    """
    camera bboxes: (N, 7) [x, y, z, h, w, l, ry]
    return lidar bboxes: (N, 7) [x_l, y_l, z_l, w, l, h, ry]
    """
    # Tr_velo_to_cam -> 4x4
    if tr_velo_to_cam.shape == (3, 4):
        tr_4x4 = np.eye(4, dtype=np.float32)
        tr_4x4[:3, :4] = tr_velo_to_cam
    elif tr_velo_to_cam.shape == (4, 4):
        tr_4x4 = tr_velo_to_cam.astype(np.float32)
    else:
        raise ValueError(f'Unexpected Tr_velo_to_cam shape: {tr_velo_to_cam.shape}')

    # R0_rect -> 4x4
    if r0_rect.shape == (3, 3):
        r0_4x4 = np.eye(4, dtype=np.float32)
        r0_4x4[:3, :3] = r0_rect
    elif r0_rect.shape == (4, 4):
        r0_4x4 = r0_rect.astype(np.float32)
    else:
        raise ValueError(f'Unexpected R0_rect shape: {r0_rect.shape}')

    # 尺寸重排 [h, w, l] -> [w, l, h]
    h = bboxes[:, 3:4]
    w = bboxes[:, 4:5]
    l = bboxes[:, 5:6]
    size_lidar = np.concatenate([w, l, h], axis=1)

    # camera -> lidar
    xyz_cam = bboxes[:, :3]
    ones = np.ones((xyz_cam.shape[0], 1), dtype=xyz_cam.dtype)
    xyz_cam_hom = np.concatenate([xyz_cam, ones], axis=1)

    cam_to_lidar = np.linalg.inv(r0_4x4 @ tr_4x4)
    xyz_lidar_hom = xyz_cam_hom @ cam_to_lidar.T
    xyz_lidar = xyz_lidar_hom[:, :3]

    ry = bboxes[:, 6:7]

    return np.concatenate([xyz_lidar, size_lidar, ry], axis=1).astype(np.float32)


class VodLidarDataset(Dataset):
    CLASSES = {
        'Pedestrian': 0,
        'Cyclist': 1,
        'Car': 2,
    }

    def __init__(
        self,
        calib_dir,
        image_dir,
        label_dir,
        velodyne_dir,
        used_classes=None,
    ):
        super().__init__()
        self.calib_dir = calib_dir
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.velodyne_dir = velodyne_dir

        if used_classes is None:
            self.class_name_to_id = self.CLASSES
        else:
            self.class_name_to_id = {
                n: self.CLASSES[n]
                for n in used_classes
                if n in self.CLASSES
            }
        self.valid_class_names = set(self.class_name_to_id.keys())

        self.sample_ids = []
        self.annos = {}
        self.calibs = {}
        self._build_index()

    def _build_index(self):
        velodyne_files = sorted(glob.glob(os.path.join(self.velodyne_dir, '*.bin')))

        for v_path in velodyne_files:
            base = os.path.splitext(os.path.basename(v_path))[0]
            calib_path = os.path.join(self.calib_dir, base + '.txt')
            label_path = os.path.join(self.label_dir, base + '.txt')
            img_jpg = os.path.join(self.image_dir, base + '.jpg')
            img_png = os.path.join(self.image_dir, base + '.png')

            if not (os.path.exists(calib_path) and os.path.exists(label_path)):
                continue

            try:
                calib = self._read_calib(calib_path)
                annos = self._read_label(label_path)
            except Exception:
                continue

            if annos is None or len(annos['name']) == 0:
                continue

            if os.path.exists(img_jpg):
                image_path = img_jpg
            elif os.path.exists(img_png):
                image_path = img_png
            else:
                image_path = None

            calib['image_path'] = image_path

            self.sample_ids.append(base)
            self.annos[base] = annos
            self.calibs[base] = calib

        print(f'[VodLidarDataset] kept {len(self.sample_ids)} frames.')

    def _read_calib(self, calib_path):
        calib = {}
        with open(calib_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                key, value = line.split(':', 1)
                value = value.strip()
                if not value:
                    continue
                vals = np.fromstring(value, sep=' ', dtype=np.float32)

                if key.startswith('P'):
                    calib[key] = vals.reshape(3, 4)
                elif key == 'R0_rect':
                    calib[key] = vals.reshape(3, 3)
                elif key == 'Tr_velo_to_cam':
                    calib[key] = vals.reshape(3, 4)
                else:
                    calib[key] = vals

        if 'R0_rect' not in calib:
            calib['R0_rect'] = np.eye(3, dtype=np.float32)
        return calib

    def _read_label(self, label_path):
        names = []
        truncated = []
        occluded = []
        alpha = []
        bbox = []
        dimensions = []
        locations = []
        rotation_y = []

        with open(label_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                parts = line.split()
                if len(parts) < 16:
                    continue

                cls = parts[0]
                if cls == 'DontCare' or cls not in self.valid_class_names:
                    continue

                names.append(cls)
                truncated.append(float(parts[1]))
                occluded.append(int(parts[2]))
                alpha.append(float(parts[3]))
                bbox.append([float(x) for x in parts[4:8]])
                dimensions.append([float(x) for x in parts[8:11]])   # h, w, l
                locations.append([float(x) for x in parts[11:14]])   # x, y, z (camera)
                rotation_y.append(float(parts[14]))                  # ry (VoD 定義)

        if len(names) == 0:
            return None

        return {
            'name': np.array(names),
            'truncated': np.array(truncated, dtype=np.float32),
            'occluded': np.array(occluded, dtype=np.int32),
            'alpha': np.array(alpha, dtype=np.float32),
            'bbox': np.array(bbox, dtype=np.float32),
            'dimensions': np.array(dimensions, dtype=np.float32),
            'location': np.array(locations, dtype=np.float32),
            'rotation_y': np.array(rotation_y, dtype=np.float32),
            'difficulty': np.zeros(len(names), dtype=np.int32),
        }

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        sample_id = self.sample_ids[index]

        velodyne_path = os.path.join(self.velodyne_dir, sample_id + '.bin')
        pts = read_points(velodyne_path)

        calib = self.calibs[sample_id]
        annos = self.annos[sample_id]

        tr_velo_to_cam = calib['Tr_velo_to_cam'].astype(np.float32)
        r0_rect = calib['R0_rect'].astype(np.float32)

        names = annos['name']
        loc = annos['location']
        dim = annos['dimensions']
        ry = annos['rotation_y']

        # [x, y, z, h, w, l, ry] (camera)
        gt_bboxes_camera = np.concatenate(
            [loc, dim, ry[:, None]],
            axis=1,
        ).astype(np.float32)

        # -> lidar
        gt_bboxes_3d = bbox_camera2lidar(gt_bboxes_camera, tr_velo_to_cam, r0_rect)

        gt_labels = np.array(
            [self.class_name_to_id[n] for n in names],
            dtype=np.int64,
        )

        return {
            'pts': pts,
            'gt_bboxes_3d': gt_bboxes_3d,
            'gt_labels': gt_labels,
            'gt_names': names,
            'difficulty': annos['difficulty'],
            'image_info': {'image_path': calib.get('image_path', None)},
            'calib_info': calib,
            'sample_id': sample_id,
        }


def vod_collate_fn(list_data):
    batched_pts = [torch.from_numpy(d['pts']) for d in list_data]
    batched_gt_bboxes = [torch.from_numpy(d['gt_bboxes_3d']) for d in list_data]
    batched_labels = [torch.from_numpy(d['gt_labels']) for d in list_data]
    batched_names = [d['gt_names'] for d in list_data]
    batched_difficulty = [torch.from_numpy(d['difficulty']) for d in list_data]
    batched_img_info = [d['image_info'] for d in list_data]
    batched_calib_info = [d['calib_info'] for d in list_data]

    return {
        'batched_pts': batched_pts,
        'batched_gt_bboxes': batched_gt_bboxes,
        'batched_labels': batched_labels,
        'batched_names': batched_names,
        'batched_difficulty': batched_difficulty,
        'batched_img_info': batched_img_info,
        'batched_calib_info': batched_calib_info,
    }


def get_vod_dataloader(
    calib_dir='view_of_delft_PUBLIC/lidar/training/calib',
    image_dir='view_of_delft_PUBLIC/lidar/training/image_2',
    label_dir='view_of_delft_PUBLIC/lidar/training/label_2',
    velodyne_dir='Enhance1024_bin',
    batch_size=8,
    num_workers=0,
    shuffle=True,
    drop_last=False,
    used_classes=None,
):
    dataset = VodLidarDataset(
        calib_dir=calib_dir,
        image_dir=image_dir,
        label_dir=label_dir,
        velodyne_dir=velodyne_dir,
        used_classes=used_classes,
    )

    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
        collate_fn=vod_collate_fn,
    )
    return dataloader, dataset


def get_vod_train_val_dataloaders(
    calib_dir='view_of_delft_PUBLIC/lidar/training/calib',
    image_dir='view_of_delft_PUBLIC/lidar/training/image_2',
    label_dir='view_of_delft_PUBLIC/lidar/training/label_2',
    velodyne_dir='view_of_delft_PUBLIC/lidar/training/velodyne',
    batch_size=4,
    num_workers=0,
    val_ratio=0.2,
    seed=42,
    used_classes=None,
):
    full_dataset = VodLidarDataset(
        calib_dir=calib_dir,
        image_dir=image_dir,
        label_dir=label_dir,
        velodyne_dir=velodyne_dir,
        used_classes=used_classes,
    )

    num_samples = len(full_dataset)
    indices = np.arange(num_samples)

    rng = np.random.RandomState(seed)
    rng.shuffle(indices)

    val_size = int(num_samples * val_ratio)
    if val_size == 0:
        raise ValueError('val_ratio 太小，validation 集為空')

    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=False,
        collate_fn=vod_collate_fn,
    )

    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=False,
        collate_fn=vod_collate_fn,
    )

    return train_loader, val_loader, train_dataset, val_dataset


if __name__ == '__main__':
    calib_dir = 'view_of_delft_PUBLIC/radar/training/calib'
    image_dir = 'view_of_delft_PUBLIC/lidar/training/image_2'
    label_dir = 'view_of_delft_PUBLIC/lidar/training/label_2'
    velodyne_dir = 'Enhance1024_bin'

    train_loader, val_loader, train_dataset, val_dataset = get_vod_train_val_dataloaders(
        calib_dir=calib_dir,
        image_dir=image_dir,
        label_dir=label_dir,
        velodyne_dir=velodyne_dir,
        batch_size=16,
        num_workers=0,
        val_ratio=0.2,
        seed=42,
        used_classes=['Car', 'Pedestrian', 'Cyclist'],
    )

    print('train samples:', len(train_dataset))
    print('val samples:', len(val_dataset))

    for batch in train_loader:
        print('[Train] pts batch size:', len(batch['batched_pts']))
        print('[Train] first frame pts shape:', batch['batched_pts'][0].shape)
        print('[Train] first frame gt bboxes:', batch['batched_gt_bboxes'][0].shape)
        break

    for batch in val_loader:
        print('[Val] pts batch size:', len(batch['batched_pts']))
        print('[Val] first frame pts shape:', batch['batched_pts'][0].shape)
        print('[Val] first frame gt bboxes:', batch['batched_gt_bboxes'][0].shape)
        break


[VodLidarDataset] kept 6285 frames.
train samples: 5028
val samples: 1257
[Train] pts batch size: 4
[Train] first frame pts shape: torch.Size([185062, 4])
[Train] first frame gt bboxes: torch.Size([7, 7])
[Val] pts batch size: 4
[Val] first frame pts shape: torch.Size([172586, 4])
[Val] first frame gt bboxes: torch.Size([9, 7])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Loss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, beta=1/9, cls_w=1.0, reg_w=2.0, dir_w=0.2):
        super().__init__()
        self.alpha = 0.25
        self.gamma = 2.0
        self.cls_w = cls_w
        self.reg_w = reg_w
        self.dir_w = dir_w
        self.smooth_l1_loss = nn.SmoothL1Loss(reduction='none',
                                              beta=beta)
        self.dir_cls = nn.CrossEntropyLoss()
    
    def forward(self,
                bbox_cls_pred,
                bbox_pred,
                bbox_dir_cls_pred,
                batched_labels, 
                num_cls_pos, 
                batched_bbox_reg, 
                batched_dir_labels):
        '''
        bbox_cls_pred: (n, 3)
        bbox_pred: (n, 7)
        bbox_dir_cls_pred: (n, 2)
        batched_labels: (n, )
        num_cls_pos: int
        batched_bbox_reg: (n, 7)
        batched_dir_labels: (n, )
        return: loss, float.
        '''
        # 1. bbox cls loss
        # focal loss: FL = - \alpha_t (1 - p_t)^\gamma * log(p_t)
        #             y == 1 -> p_t = p
        #             y == 0 -> p_t = 1 - p
        nclasses = bbox_cls_pred.size(1)
        batched_labels = F.one_hot(batched_labels, nclasses + 1)[:, :nclasses].float() # (n, 3)

        bbox_cls_pred_sigmoid = torch.sigmoid(bbox_cls_pred)
        weights = self.alpha * (1 - bbox_cls_pred_sigmoid).pow(self.gamma) * batched_labels + \
             (1 - self.alpha) * bbox_cls_pred_sigmoid.pow(self.gamma) * (1 - batched_labels) # (n, 3)
        cls_loss = F.binary_cross_entropy(bbox_cls_pred_sigmoid, batched_labels, reduction='none')
        cls_loss = cls_loss * weights
        cls_loss = cls_loss.sum() / num_cls_pos
        
        # 2. regression loss
        reg_loss = self.smooth_l1_loss(bbox_pred, batched_bbox_reg)
        reg_loss = reg_loss.sum() / reg_loss.size(0)

        # 3. direction cls loss
        dir_cls_loss = self.dir_cls(bbox_dir_cls_pred, batched_dir_labels)

        # 4. total loss
        total_loss = self.cls_w * cls_loss + self.reg_w * reg_loss + self.dir_w * dir_cls_loss
        
        loss_dict={'cls_loss': cls_loss, 
                   'reg_loss': reg_loss,
                   'dir_cls_loss': dir_cls_loss,
                   'total_loss': total_loss}
        return loss_dict
    

import numpy as np
import pdb
import torch
import torch.nn as nn
import torch.nn.functional as F
from pointpillars.model.anchors import Anchors, anchor_target, anchors2bboxes
from pointpillars.ops import Voxelization, nms_cuda
from pointpillars.utils import limit_period


class PillarLayer(nn.Module):
    def __init__(self, voxel_size, point_cloud_range, max_num_points, max_voxels):
        super().__init__()
        self.voxel_layer = Voxelization(voxel_size=voxel_size,
                                        point_cloud_range=point_cloud_range,
                                        max_num_points=max_num_points,
                                        max_voxels=max_voxels)

    @torch.no_grad()
    def forward(self, batched_pts):
        '''
        batched_pts: list[tensor], len(batched_pts) = bs
        return: 
               pillars: (p1 + p2 + ... + pb, num_points, c), 
               coors_batch: (p1 + p2 + ... + pb, 1 + 3), 
               num_points_per_pillar: (p1 + p2 + ... + pb, ), (b: batch size)
        '''
        pillars, coors, npoints_per_pillar = [], [], []
        for i, pts in enumerate(batched_pts):
            voxels_out, coors_out, num_points_per_voxel_out = self.voxel_layer(pts) 
            # voxels_out: (max_voxel, num_points, c), coors_out: (max_voxel, 3)
            # num_points_per_voxel_out: (max_voxel, )
            pillars.append(voxels_out)
            coors.append(coors_out.long())
            npoints_per_pillar.append(num_points_per_voxel_out)
        
        pillars = torch.cat(pillars, dim=0) # (p1 + p2 + ... + pb, num_points, c)
        npoints_per_pillar = torch.cat(npoints_per_pillar, dim=0) # (p1 + p2 + ... + pb, )
        coors_batch = []
        for i, cur_coors in enumerate(coors):
            coors_batch.append(F.pad(cur_coors, (1, 0), value=i))
        coors_batch = torch.cat(coors_batch, dim=0) # (p1 + p2 + ... + pb, 1 + 3)

        return pillars, coors_batch, npoints_per_pillar


class PillarEncoder(nn.Module):
    def __init__(self, voxel_size, point_cloud_range, in_channel, out_channel):
        super().__init__()
        self.out_channel = out_channel
        self.vx, self.vy = voxel_size[0], voxel_size[1]
        self.x_offset = voxel_size[0] / 2 + point_cloud_range[0]
        self.y_offset = voxel_size[1] / 2 + point_cloud_range[1]
        self.x_l = int((point_cloud_range[3] - point_cloud_range[0]) / voxel_size[0])
        self.y_l = int((point_cloud_range[4] - point_cloud_range[1]) / voxel_size[1])

        self.conv = nn.Conv1d(in_channel, out_channel, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channel, eps=1e-3, momentum=0.01)

    def forward(self, pillars, coors_batch, npoints_per_pillar):
        '''
        pillars: (p1 + p2 + ... + pb, num_points, c), c = 4
        coors_batch: (p1 + p2 + ... + pb, 1 + 3)
        npoints_per_pillar: (p1 + p2 + ... + pb, )
        return:  (bs, out_channel, y_l, x_l)
        '''
        device = pillars.device
        # 1. calculate offset to the points center (in each pillar)
        offset_pt_center = pillars[:, :, :3] - torch.sum(pillars[:, :, :3], dim=1, keepdim=True) / npoints_per_pillar[:, None, None] # (p1 + p2 + ... + pb, num_points, 3)

        # 2. calculate offset to the pillar center
        x_offset_pi_center = pillars[:, :, :1] - (coors_batch[:, None, 1:2] * self.vx + self.x_offset) # (p1 + p2 + ... + pb, num_points, 1)
        y_offset_pi_center = pillars[:, :, 1:2] - (coors_batch[:, None, 2:3] * self.vy + self.y_offset) # (p1 + p2 + ... + pb, num_points, 1)

        # 3. encoder
        features = torch.cat([pillars, offset_pt_center, x_offset_pi_center, y_offset_pi_center], dim=-1) # (p1 + p2 + ... + pb, num_points, 9)
        features[:, :, 0:1] = x_offset_pi_center # tmp
        features[:, :, 1:2] = y_offset_pi_center # tmp
        # In consitent with mmdet3d. 
        # The reason can be referenced to https://github.com/open-mmlab/mmdetection3d/issues/1150

        # 4. find mask for (0, 0, 0) and update the encoded features
        # a very beautiful implementation
        voxel_ids = torch.arange(0, pillars.size(1)).to(device) # (num_points, )
        mask = voxel_ids[:, None] < npoints_per_pillar[None, :] # (num_points, p1 + p2 + ... + pb)
        mask = mask.permute(1, 0).contiguous()  # (p1 + p2 + ... + pb, num_points)
        features *= mask[:, :, None]

        # 5. embedding
        features = features.permute(0, 2, 1).contiguous() # (p1 + p2 + ... + pb, 9, num_points)
        features = F.relu(self.bn(self.conv(features)))  # (p1 + p2 + ... + pb, out_channels, num_points)
        pooling_features = torch.max(features, dim=-1)[0] # (p1 + p2 + ... + pb, out_channels)

        # 6. pillar scatter
        batched_canvas = []
        bs = coors_batch[-1, 0] + 1
        for i in range(bs):
            cur_coors_idx = coors_batch[:, 0] == i
            cur_coors = coors_batch[cur_coors_idx, :]
            cur_features = pooling_features[cur_coors_idx]

            canvas = torch.zeros((self.x_l, self.y_l, self.out_channel), dtype=torch.float32, device=device)
            canvas[cur_coors[:, 1], cur_coors[:, 2]] = cur_features
            canvas = canvas.permute(2, 1, 0).contiguous()
            batched_canvas.append(canvas)
        batched_canvas = torch.stack(batched_canvas, dim=0) # (bs, in_channel, self.y_l, self.x_l)
        return batched_canvas


class Backbone(nn.Module):
    def __init__(self, in_channel, out_channels, layer_nums, layer_strides=[2, 2, 2]):
        super().__init__()
        assert len(out_channels) == len(layer_nums)
        assert len(out_channels) == len(layer_strides)
        
        self.multi_blocks = nn.ModuleList()
        for i in range(len(layer_strides)):
            blocks = []
            blocks.append(nn.Conv2d(in_channel, out_channels[i], 3, stride=layer_strides[i], bias=False, padding=1))
            blocks.append(nn.BatchNorm2d(out_channels[i], eps=1e-3, momentum=0.01))
            blocks.append(nn.ReLU(inplace=True))

            for _ in range(layer_nums[i]):
                blocks.append(nn.Conv2d(out_channels[i], out_channels[i], 3, bias=False, padding=1))
                blocks.append(nn.BatchNorm2d(out_channels[i], eps=1e-3, momentum=0.01))
                blocks.append(nn.ReLU(inplace=True))

            in_channel = out_channels[i]
            self.multi_blocks.append(nn.Sequential(*blocks))

        # in consitent with mmdet3d
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        '''
        x: (b, c, y_l, x_l). Default: (6, 64, 496, 432)
        return: list[]. Default: [(6, 64, 248, 216), (6, 128, 124, 108), (6, 256, 62, 54)]
        '''
        outs = []
        for i in range(len(self.multi_blocks)):
            x = self.multi_blocks[i](x)
            outs.append(x)
        return outs


class Neck(nn.Module):
    def __init__(self, in_channels, upsample_strides, out_channels):
        super().__init__()
        assert len(in_channels) == len(upsample_strides)
        assert len(upsample_strides) == len(out_channels)

        self.decoder_blocks = nn.ModuleList()
        for i in range(len(in_channels)):
            decoder_block = []
            decoder_block.append(nn.ConvTranspose2d(in_channels[i], 
                                                    out_channels[i], 
                                                    upsample_strides[i], 
                                                    stride=upsample_strides[i],
                                                    bias=False))
            decoder_block.append(nn.BatchNorm2d(out_channels[i], eps=1e-3, momentum=0.01))
            decoder_block.append(nn.ReLU(inplace=True))

            self.decoder_blocks.append(nn.Sequential(*decoder_block))
        
        # in consitent with mmdet3d
        for m in self.modules():
            if isinstance(m, nn.ConvTranspose2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        '''
        x: [(bs, 64, 248, 216), (bs, 128, 124, 108), (bs, 256, 62, 54)]
        return: (bs, 384, 248, 216)
        '''
        outs = []
        for i in range(len(self.decoder_blocks)):
            xi = self.decoder_blocks[i](x[i]) # (bs, 128, 248, 216)
            outs.append(xi)
        out = torch.cat(outs, dim=1)
        return out


class Head(nn.Module):
    def __init__(self, in_channel, n_anchors, n_classes):
        super().__init__()
        
        self.conv_cls = nn.Conv2d(in_channel, n_anchors*n_classes, 1)
        self.conv_reg = nn.Conv2d(in_channel, n_anchors*7, 1)
        self.conv_dir_cls = nn.Conv2d(in_channel, n_anchors*2, 1)

        # in consitent with mmdet3d
        conv_layer_id = 0
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, mean=0, std=0.01)
                if conv_layer_id == 0:
                    prior_prob = 0.01
                    bias_init = float(-np.log((1 - prior_prob) / prior_prob))
                    nn.init.constant_(m.bias, bias_init)
                else:
                    nn.init.constant_(m.bias, 0)
                conv_layer_id += 1

    def forward(self, x):
        '''
        x: (bs, 384, 248, 216)
        return: 
              bbox_cls_pred: (bs, n_anchors*3, 248, 216) 
              bbox_pred: (bs, n_anchors*7, 248, 216)
              bbox_dir_cls_pred: (bs, n_anchors*2, 248, 216)
        '''
        bbox_cls_pred = self.conv_cls(x)
        bbox_pred = self.conv_reg(x)
        bbox_dir_cls_pred = self.conv_dir_cls(x)
        return bbox_cls_pred, bbox_pred, bbox_dir_cls_pred


class PointPillars(nn.Module):
    def __init__(self,
                 nclasses=3, 
                 voxel_size=[0.16, 0.16, 4],
                 point_cloud_range=[0, -39.68, -3, 69.12, 39.68, 1],
                 max_num_points=32,
                 max_voxels=(16000, 40000)):
        super().__init__()
        self.nclasses = nclasses
        self.pillar_layer = PillarLayer(voxel_size=voxel_size, 
                                        point_cloud_range=point_cloud_range, 
                                        max_num_points=max_num_points, 
                                        max_voxels=max_voxels)
        self.pillar_encoder = PillarEncoder(voxel_size=voxel_size, 
                                            point_cloud_range=point_cloud_range, 
                                            in_channel=9, 
                                            out_channel=64)
        self.backbone = Backbone(in_channel=64, 
                                 out_channels=[64, 128, 256], 
                                 layer_nums=[3, 5, 5])
        self.neck = Neck(in_channels=[64, 128, 256], 
                         upsample_strides=[1, 2, 4], 
                         out_channels=[128, 128, 128])
        self.head = Head(in_channel=384, n_anchors=2*nclasses, n_classes=nclasses)
        
        # anchors
        ranges = [[0, -39.68, -0.6, 69.12, 39.68, -0.6],
                    [0, -39.68, -0.6, 69.12, 39.68, -0.6],
                    [0, -39.68, -1.78, 69.12, 39.68, -1.78]]
        sizes = [[0.6, 0.8, 1.73], [0.6, 1.76, 1.73], [1.6, 3.9, 1.56]]
        rotations=[0, 1.57]
        self.anchors_generator = Anchors(ranges=ranges, 
                                         sizes=sizes, 
                                         rotations=rotations)
        
        # train
        self.assigners = [
            {'pos_iou_thr': 0.5, 'neg_iou_thr': 0.35, 'min_iou_thr': 0.35},
            {'pos_iou_thr': 0.5, 'neg_iou_thr': 0.35, 'min_iou_thr': 0.35},
            {'pos_iou_thr': 0.6, 'neg_iou_thr': 0.45, 'min_iou_thr': 0.45},
        ]

        # val and test
        self.nms_pre = 100
        self.nms_thr = 0.01
        self.score_thr = 0.1
        self.max_num = 50

    def get_predicted_bboxes_single(self, bbox_cls_pred, bbox_pred, bbox_dir_cls_pred, anchors):
        '''
        bbox_cls_pred: (n_anchors*3, 248, 216) 
        bbox_pred: (n_anchors*7, 248, 216)
        bbox_dir_cls_pred: (n_anchors*2, 248, 216)
        anchors: (y_l, x_l, 3, 2, 7)
        return: 
            bboxes: (k, 7)
            labels: (k, )
            scores: (k, ) 
        '''
        # 0. pre-process 
        bbox_cls_pred = bbox_cls_pred.permute(1, 2, 0).reshape(-1, self.nclasses)
        bbox_pred = bbox_pred.permute(1, 2, 0).reshape(-1, 7)
        bbox_dir_cls_pred = bbox_dir_cls_pred.permute(1, 2, 0).reshape(-1, 2)
        anchors = anchors.reshape(-1, 7)
        
        bbox_cls_pred = torch.sigmoid(bbox_cls_pred)
        bbox_dir_cls_pred = torch.max(bbox_dir_cls_pred, dim=1)[1]

        # 1. obtain self.nms_pre bboxes based on scores
        inds = bbox_cls_pred.max(1)[0].topk(self.nms_pre)[1]
        bbox_cls_pred = bbox_cls_pred[inds]
        bbox_pred = bbox_pred[inds]
        bbox_dir_cls_pred = bbox_dir_cls_pred[inds]
        anchors = anchors[inds]

        # 2. decode predicted offsets to bboxes
        bbox_pred = anchors2bboxes(anchors, bbox_pred)

        # 3. nms
        bbox_pred2d_xy = bbox_pred[:, [0, 1]]
        bbox_pred2d_lw = bbox_pred[:, [3, 4]]
        bbox_pred2d = torch.cat([bbox_pred2d_xy - bbox_pred2d_lw / 2,
                                 bbox_pred2d_xy + bbox_pred2d_lw / 2,
                                 bbox_pred[:, 6:]], dim=-1) # (n_anchors, 5)

        ret_bboxes, ret_labels, ret_scores = [], [], []
        for i in range(self.nclasses):
            # 3.1 filter bboxes with scores below self.score_thr
            cur_bbox_cls_pred = bbox_cls_pred[:, i]
            score_inds = cur_bbox_cls_pred > self.score_thr
            if score_inds.sum() == 0:
                continue

            cur_bbox_cls_pred = cur_bbox_cls_pred[score_inds]
            cur_bbox_pred2d = bbox_pred2d[score_inds]
            cur_bbox_pred = bbox_pred[score_inds]
            cur_bbox_dir_cls_pred = bbox_dir_cls_pred[score_inds]
            
            # 3.2 nms core
            keep_inds = nms_cuda(boxes=cur_bbox_pred2d, 
                                 scores=cur_bbox_cls_pred, 
                                 thresh=self.nms_thr, 
                                 pre_maxsize=None, 
                                 post_max_size=None)

            cur_bbox_cls_pred = cur_bbox_cls_pred[keep_inds]
            cur_bbox_pred = cur_bbox_pred[keep_inds]
            cur_bbox_dir_cls_pred = cur_bbox_dir_cls_pred[keep_inds]
            cur_bbox_pred[:, -1] = limit_period(cur_bbox_pred[:, -1].detach().cpu(), 1, np.pi).to(cur_bbox_pred) # [-pi, 0]
            cur_bbox_pred[:, -1] += (1 - cur_bbox_dir_cls_pred) * np.pi

            ret_bboxes.append(cur_bbox_pred)
            ret_labels.append(torch.zeros_like(cur_bbox_pred[:, 0], dtype=torch.long) + i)
            ret_scores.append(cur_bbox_cls_pred)

        # 4. filter some bboxes if bboxes number is above self.max_num
        if len(ret_bboxes) == 0:
            return [], [], []
        ret_bboxes = torch.cat(ret_bboxes, 0)
        ret_labels = torch.cat(ret_labels, 0)
        ret_scores = torch.cat(ret_scores, 0)
        if ret_bboxes.size(0) > self.max_num:
            final_inds = ret_scores.topk(self.max_num)[1]
            ret_bboxes = ret_bboxes[final_inds]
            ret_labels = ret_labels[final_inds]
            ret_scores = ret_scores[final_inds]
        result = {
            'lidar_bboxes': ret_bboxes.detach().cpu().numpy(),
            'labels': ret_labels.detach().cpu().numpy(),
            'scores': ret_scores.detach().cpu().numpy()
        }
        return result


    def get_predicted_bboxes(self, bbox_cls_pred, bbox_pred, bbox_dir_cls_pred, batched_anchors):
        '''
        bbox_cls_pred: (bs, n_anchors*3, 248, 216) 
        bbox_pred: (bs, n_anchors*7, 248, 216)
        bbox_dir_cls_pred: (bs, n_anchors*2, 248, 216)
        batched_anchors: (bs, y_l, x_l, 3, 2, 7)
        return: 
            bboxes: [(k1, 7), (k2, 7), ... ]
            labels: [(k1, ), (k2, ), ... ]
            scores: [(k1, ), (k2, ), ... ] 
        '''
        results = []
        bs = bbox_cls_pred.size(0)
        for i in range(bs):
            result = self.get_predicted_bboxes_single(bbox_cls_pred=bbox_cls_pred[i],
                                                      bbox_pred=bbox_pred[i], 
                                                      bbox_dir_cls_pred=bbox_dir_cls_pred[i], 
                                                      anchors=batched_anchors[i])
            results.append(result)
        return results

    def forward(self, batched_pts, mode='test', batched_gt_bboxes=None, batched_gt_labels=None):
        batch_size = len(batched_pts)
        # batched_pts: list[tensor] -> pillars: (p1 + p2 + ... + pb, num_points, c), 
        #                              coors_batch: (p1 + p2 + ... + pb, 1 + 3), 
        #                              num_points_per_pillar: (p1 + p2 + ... + pb, ), (b: batch size)
        pillars, coors_batch, npoints_per_pillar = self.pillar_layer(batched_pts)

        # pillars: (p1 + p2 + ... + pb, num_points, c), c = 4
        # coors_batch: (p1 + p2 + ... + pb, 1 + 3)
        # npoints_per_pillar: (p1 + p2 + ... + pb, )
        #                     -> pillar_features: (bs, out_channel, y_l, x_l)
        pillar_features = self.pillar_encoder(pillars, coors_batch, npoints_per_pillar)

        # xs:  [(bs, 64, 248, 216), (bs, 128, 124, 108), (bs, 256, 62, 54)]
        xs = self.backbone(pillar_features)

        # x: (bs, 384, 248, 216)
        x = self.neck(xs)

        # bbox_cls_pred: (bs, n_anchors*3, 248, 216) 
        # bbox_pred: (bs, n_anchors*7, 248, 216)
        # bbox_dir_cls_pred: (bs, n_anchors*2, 248, 216)
        bbox_cls_pred, bbox_pred, bbox_dir_cls_pred = self.head(x)

        # anchors
        device = bbox_cls_pred.device
        feature_map_size = torch.tensor(list(bbox_cls_pred.size()[-2:]), device=device)
        anchors = self.anchors_generator.get_multi_anchors(feature_map_size)
        batched_anchors = [anchors for _ in range(batch_size)]

        if mode == 'train':
            anchor_target_dict = anchor_target(batched_anchors=batched_anchors, 
                                               batched_gt_bboxes=batched_gt_bboxes, 
                                               batched_gt_labels=batched_gt_labels, 
                                               assigners=self.assigners,
                                               nclasses=self.nclasses)
            
            return bbox_cls_pred, bbox_pred, bbox_dir_cls_pred, anchor_target_dict
        elif mode == 'val':
            results = self.get_predicted_bboxes(bbox_cls_pred=bbox_cls_pred, 
                                                bbox_pred=bbox_pred, 
                                                bbox_dir_cls_pred=bbox_dir_cls_pred, 
                                                batched_anchors=batched_anchors)
            return results

        elif mode == 'test':
            results = self.get_predicted_bboxes(bbox_cls_pred=bbox_cls_pred, 
                                                bbox_pred=bbox_pred, 
                                                bbox_dir_cls_pred=bbox_dir_cls_pred, 
                                                batched_anchors=batched_anchors)
            return results
        else:
            raise ValueError   


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# train_pointpillars.py
import os
import torch
from tqdm import tqdm


def _move_batch_to_device(data_dict, device):
    for key in data_dict:
        value = data_dict[key]
        if isinstance(value, list):
            for i, item in enumerate(value):
                if torch.is_tensor(item):
                    data_dict[key][i] = item.to(device)
    return data_dict


def _loss_dict_to_float(loss_dict):
    out = {}
    for k, v in loss_dict.items():
        out[k] = float(v.detach().cpu().item()) if torch.is_tensor(v) else float(v)
    return out


def train_pointpillars(
    train_dataloader,
    val_dataloader=None,
    num_classes=3,
    max_epoch=80,
    init_lr=2e-4,
    saved_path="./output",
    ckpt_freq_epoch=1,
    early_stop_patience=10,     # <<<<<<< 多加這個
    use_cuda=True,
):
    device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")

    pointpillars = PointPillars(nclasses=num_classes).to(device)
    loss_func = Loss()

    max_iters = len(train_dataloader) * max_epoch
    optimizer = torch.optim.AdamW(
        params=pointpillars.parameters(),
        lr=init_lr,
        betas=(0.95, 0.99),
        weight_decay=0.01,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=init_lr * 10,
        total_steps=max_iters,
        pct_start=0.4,
        anneal_strategy="cos",
        cycle_momentum=True,
        base_momentum=0.895 * 0.95,
        max_momentum=0.95,
        div_factor=10,
    )

    saved_ckpt_path = os.path.join(saved_path, "checkpoints")
    os.makedirs(saved_ckpt_path, exist_ok=True)

    best_val_loss = float("inf")
    best_epoch = -1
    best_ckpt_file = None

    # *** Early Stopping 記錄變數 ***
    patience_counter = 0

    # ===========================================================
    #                     Training Loop
    # ===========================================================

    for epoch in range(max_epoch):
        print("=" * 30, f"Epoch {epoch+1}/{max_epoch}", "=" * 30)

        # --------------- TRAIN ----------------
        pointpillars.train()
        train_loss_sum = {}
        train_steps = 0

        pbar = tqdm(train_dataloader, desc=f"[Train] {epoch+1}", ncols=120)
        for data_dict in pbar:
            data_dict = _move_batch_to_device(data_dict, device)

            optimizer.zero_grad()
            batched_pts = data_dict["batched_pts"]
            batched_gt_bboxes = data_dict["batched_gt_bboxes"]
            batched_labels = data_dict["batched_labels"]

            (
                bbox_cls_pred,
                bbox_pred,
                bbox_dir_cls_pred,
                anchor_target_dict,
            ) = pointpillars(
                batched_pts=batched_pts,
                mode="train",
                batched_gt_bboxes=batched_gt_bboxes,
                batched_gt_labels=batched_labels,
            )

            bbox_cls_pred = bbox_cls_pred.permute(0, 2, 3, 1).reshape(-1, num_classes)
            bbox_pred = bbox_pred.permute(0, 2, 3, 1).reshape(-1, 7)
            bbox_dir_cls_pred = bbox_dir_cls_pred.permute(0, 2, 3, 1).reshape(-1, 2)

            batched_bbox_labels = anchor_target_dict["batched_labels"].reshape(-1)
            batched_label_weights = anchor_target_dict["batched_label_weights"].reshape(-1)
            batched_bbox_reg = anchor_target_dict["batched_bbox_reg"].reshape(-1, 7)
            batched_dir_labels = anchor_target_dict["batched_dir_labels"].reshape(-1)

            pos_idx = (batched_bbox_labels >= 0) & (batched_bbox_labels < num_classes)

            bbox_pred = bbox_pred[pos_idx]
            batched_bbox_reg = batched_bbox_reg[pos_idx]

            bbox_pred_angle = bbox_pred[:, -1].clone()
            reg_angle = batched_bbox_reg[:, -1].clone()
            bbox_pred[:, -1] = torch.sin(bbox_pred_angle) * torch.cos(reg_angle)
            batched_bbox_reg[:, -1] = torch.cos(bbox_pred_angle) * torch.sin(reg_angle)

            bbox_dir_cls_pred = bbox_dir_cls_pred[pos_idx]
            batched_dir_labels = batched_dir_labels[pos_idx]

            num_cls_pos = (batched_bbox_labels < num_classes).sum()

            bbox_cls_pred = bbox_cls_pred[batched_label_weights > 0]
            batched_bbox_labels[batched_bbox_labels < 0] = num_classes
            batched_bbox_labels = batched_bbox_labels[batched_label_weights > 0]

            loss_dict = loss_func(
                bbox_cls_pred=bbox_cls_pred,
                bbox_pred=bbox_pred,
                bbox_dir_cls_pred=bbox_dir_cls_pred,
                batched_labels=batched_bbox_labels,
                num_cls_pos=num_cls_pos,
                batched_bbox_reg=batched_bbox_reg,
                batched_dir_labels=batched_dir_labels,
            )

            loss = loss_dict["total_loss"]
            loss.backward()
            optimizer.step()
            scheduler.step()

            loss_dict_float = _loss_dict_to_float(loss_dict)
            cur_lr = scheduler.get_last_lr()[0]

            pbar.set_postfix(loss=f"{loss_dict_float['total_loss']:.4f}", lr=f"{cur_lr:.6f}")

            for k, v in loss_dict_float.items():
                train_loss_sum[k] = train_loss_sum.get(k, 0.0) + v

            train_steps += 1

        print(f"[Train] Epoch {epoch+1} | "
              + ", ".join([f"{k}:{v/train_steps:.5f}" for k, v in train_loss_sum.items()]))

        # --------------- SAVE EPOCH CKPT ----------------
        if (epoch + 1) % ckpt_freq_epoch == 0:
            ckpt_file = os.path.join(saved_ckpt_path, f"epoch_{epoch+1}.pth")
            torch.save(pointpillars.state_dict(), ckpt_file)
            print(f"Checkpoint saved: {ckpt_file}")

        # No val => 不做 early stopping
        if val_dataloader is None:
            continue

        # --------------- VALID ----------------
        pointpillars.eval()
        val_loss_sum = {}
        val_steps = 0

        with torch.no_grad():
            pbar_val = tqdm(val_dataloader, desc=f"[Val] {epoch+1}", ncols=120)
            for data_dict in pbar_val:
                data_dict = _move_batch_to_device(data_dict, device)

                batched_pts = data_dict["batched_pts"]
                batched_gt_bboxes = data_dict["batched_gt_bboxes"]
                batched_labels = data_dict["batched_labels"]

                (
                    bbox_cls_pred,
                    bbox_pred,
                    bbox_dir_cls_pred,
                    anchor_target_dict
                ) = pointpillars(
                    batched_pts=batched_pts,
                    mode="train",
                    batched_gt_bboxes=batched_gt_bboxes,
                    batched_gt_labels=batched_labels,
                )

                bbox_cls_pred = bbox_cls_pred.permute(0, 2, 3, 1).reshape(-1, num_classes)
                bbox_pred = bbox_pred.permute(0, 2, 3, 1).reshape(-1, 7)
                bbox_dir_cls_pred = bbox_dir_cls_pred.permute(0, 2, 3, 1).reshape(-1, 2)

                batched_bbox_labels = anchor_target_dict["batched_labels"].reshape(-1)
                batched_label_weights = anchor_target_dict["batched_label_weights"].reshape(-1)
                batched_bbox_reg = anchor_target_dict["batched_bbox_reg"].reshape(-1, 7)
                batched_dir_labels = anchor_target_dict["batched_dir_labels"].reshape(-1)

                pos_idx = (batched_bbox_labels >= 0) & (batched_bbox_labels < num_classes)

                bbox_pred = bbox_pred[pos_idx]
                batched_bbox_reg = batched_bbox_reg[pos_idx]

                bbox_pred_angle = bbox_pred[:, -1].clone()
                reg_angle = batched_bbox_reg[:, -1].clone()
                bbox_pred[:, -1] = torch.sin(bbox_pred_angle) * torch.cos(reg_angle)
                batched_bbox_reg[:, -1] = torch.cos(bbox_pred_angle) * torch.sin(reg_angle)

                bbox_dir_cls_pred = bbox_dir_cls_pred[pos_idx]
                batched_dir_labels = batched_dir_labels[pos_idx]

                num_cls_pos = (batched_bbox_labels < num_classes).sum()

                bbox_cls_pred = bbox_cls_pred[batched_label_weights > 0]
                batched_bbox_labels[batched_bbox_labels < 0] = num_classes
                batched_bbox_labels = batched_bbox_labels[batched_label_weights > 0]

                loss_dict = loss_func(
                    bbox_cls_pred=bbox_cls_pred,
                    bbox_pred=bbox_pred,
                    bbox_dir_cls_pred=bbox_dir_cls_pred,
                    batched_labels=batched_bbox_labels,
                    num_cls_pos=num_cls_pos,
                    batched_bbox_reg=batched_bbox_reg,
                    batched_dir_labels=batched_dir_labels,
                )

                loss_dict_float = _loss_dict_to_float(loss_dict)

                pbar_val.set_postfix(loss=f"{loss_dict_float['total_loss']:.4f}")

                for k, v in loss_dict_float.items():
                    val_loss_sum[k] = val_loss_sum.get(k, 0.0) + v
                val_steps += 1

        val_epoch_loss = {k: v / val_steps for k, v in val_loss_sum.items()}
        print(f"[Val] Epoch {epoch+1} | "
              + ", ".join([f"{k}:{v:.5f}" for k, v in val_epoch_loss.items()]))

        # =========================
        #   ⏳ EARLY STOPPING HERE
        # =========================
        val_loss = val_epoch_loss["total_loss"]

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0

            best_ckpt_file = os.path.join(saved_ckpt_path, "best_model.pth")
            torch.save(pointpillars.state_dict(), best_ckpt_file)

            print(f"[BEST] epoch={best_epoch}  loss={best_val_loss:.6f}")
        else:
            patience_counter += 1
            print(f"[EarlyStop] No improvement {patience_counter}/{early_stop_patience}")

        if patience_counter >= early_stop_patience:
            print(f"EARLY STOPPING TRIGGERED at epoch {epoch+1}")
            break

        pointpillars.train()

    if best_ckpt_file is not None:
        print(f"Loading best model from epoch {best_epoch}")
        state_dict = torch.load(best_ckpt_file, map_location=device)
        pointpillars.load_state_dict(state_dict)

    return pointpillars


if __name__ == "__main__":

    model = train_pointpillars(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        num_classes=3,
        max_epoch=80,
        init_lr=2e-4,
        saved_path="./runs_pointpillars",
        ckpt_freq_epoch=1,
        early_stop_patience=3,   # ⏳ 想提早停更快改小
        use_cuda=True,
    )
    pass


============================== Epoch 1/80 ==============================


[Train] 1:   0%|                                                                               | 0/1257 [00:00<?, ?it/s]c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4316.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[Train] 1: 100%|██████████████████████████████████████████| 1257/1257 [06:47<00:00,  3.09it/s, loss=0.9380, lr=0.000204]


[Train] Epoch 1 | cls_loss:0.58554, reg_loss:0.56163, dir_cls_loss:0.49931, total_loss:1.80867
Checkpoint saved: ./runs_pointpillars\checkpoints\epoch_1.pth


[Val] 1: 100%|███████████████████████████████████████████████████████████| 315/315 [01:32<00:00,  3.41it/s, loss=1.3082]


[Val] Epoch 1 | cls_loss:0.43030, reg_loss:0.56611, dir_cls_loss:0.40373, total_loss:1.64327
[BEST] epoch=1  loss=1.643273
============================== Epoch 2/80 ==============================


...

[Val] Epoch 15 | cls_loss:0.16733, reg_loss:0.18910, dir_cls_loss:0.06985, total_loss:0.55949
[EarlyStop] No improvement 2/3
============================== Epoch 16/80 ==============================


[Train] 16: 100%|█████████████████████████████████████████| 1257/1257 [05:55<00:00,  3.54it/s, loss=0.4092, lr=0.001100]


[Train] Epoch 16 | cls_loss:0.11394, reg_loss:0.12731, dir_cls_loss:0.03836, total_loss:0.37622
Checkpoint saved: ./runs_pointpillars\checkpoints\epoch_16.pth


[Val] 16: 100%|██████████████████████████████████████████████████████████| 315/315 [01:15<00:00,  4.18it/s, loss=0.4363]

[Val] Epoch 16 | cls_loss:0.18200, reg_loss:0.31761, dir_cls_loss:0.09044, total_loss:0.83530
[EarlyStop] No improvement 3/3
EARLY STOPPING TRIGGERED at epoch 16
Loading best model from epoch 13


In [5]:
# eval_pointpillars_map.py

import os
import numpy as np
import torch
from tqdm import tqdm


###############################################
# 若這個檔案與 train_pointpillars.py 在同一個檔案中
# 下面這個 _move_batch_to_device 可以直接用你上面的定義
###############################################
def _move_batch_to_device(data_dict, device):
    for key in data_dict:
        value = data_dict[key]
        if isinstance(value, list):
            for i, item in enumerate(value):
                if torch.is_tensor(item):
                    data_dict[key][i] = item.to(device)
    return data_dict


###############################################
# BEV IoU（axis-aligned），與 NMS 使用方式一致
###############################################
def boxes_to_bev(boxes):
    """
    boxes: (N, 7) -> [x, y, z, w, l, h, yaw]
    回傳: (N, 4) -> [x1, y1, x2, y2]
    """
    x = boxes[:, 0]
    y = boxes[:, 1]
    w = boxes[:, 3]
    l = boxes[:, 4]
    x1 = x - w / 2.0
    y1 = y - l / 2.0
    x2 = x + w / 2.0
    y2 = y + l / 2.0
    return np.stack([x1, y1, x2, y2], axis=1).astype(np.float32)


def bev_iou(boxes1, boxes2):
    """
    boxes1: (N, 4) [x1,y1,x2,y2]
    boxes2: (M, 4)
    回傳 IoU: (N, M)
    """
    if boxes1.shape[0] == 0 or boxes2.shape[0] == 0:
        return np.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=np.float32)

    ious = np.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=np.float32)

    for i in range(boxes1.shape[0]):
        x1, y1, x2, y2 = boxes1[i]
        area1 = max(0.0, x2 - x1) * max(0.0, y2 - y1)
        if area1 <= 0:
            continue

        xx1 = np.maximum(x1, boxes2[:, 0])
        yy1 = np.maximum(y1, boxes2[:, 1])
        xx2 = np.minimum(x2, boxes2[:, 2])
        yy2 = np.minimum(y2, boxes2[:, 3])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h

        area2 = np.maximum(0.0, boxes2[:, 2] - boxes2[:, 0]) * \
                np.maximum(0.0, boxes2[:, 3] - boxes2[:, 1])
        union = area1 + area2 - inter
        iou = np.where(union > 0, inter / union, 0.0)

        ious[i, :] = iou

    return ious


###############################################
# PASCAL VOC style AP 計算
###############################################
def compute_ap(recall, precision):
    """
    recall, precision: 1D numpy array (同長度，score 由高到低的累積曲線)
    回傳 AP (scalar)
    """
    if recall.size == 0:
        return 0.0

    # 在兩端補點
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    # 做 precision envelope
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    # 只在 recall 有變化的地方累積面積
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return float(ap)


###############################################
# 主評估函式：計算每個 class 的 AP 以及 mAP
# 支援「每個類別不同 IoU 閾值」
###############################################
def _normalize_iou_thresholds(iou_thresholds, num_classes, default_thr=0.5):
    """
    將 iou_thresholds 轉成長度為 num_classes 的 list[float]
    支援:
      - 單一 float/int: 所有類別相同 IoU
      - list/tuple: 每個類別一個，長度需 == num_classes
      - dict: key=class_id, value=iou_threshold，沒指定的類別用 default_thr
    """
    if isinstance(iou_thresholds, (float, int)):
        return [float(iou_thresholds)] * num_classes
    elif isinstance(iou_thresholds, dict):
        thr_list = []
        for c in range(num_classes):
            thr_list.append(float(iou_thresholds.get(c, default_thr)))
        return thr_list
    else:
        # 假設是 list / tuple
        thr_list = list(iou_thresholds)
        if len(thr_list) != num_classes:
            raise ValueError(
                f"iou_thresholds 長度 ({len(thr_list)}) 必須等於 num_classes ({num_classes})"
            )
        return [float(t) for t in thr_list]


def evaluate_pointpillars_map(
    model,
    dataloader,
    num_classes=3,
    iou_thresholds=0.5,
    device=None,
    class_names=None,
):
    """
    model: 已訓練好的 PointPillars (或載入 checkpoint 的 model)
    dataloader: 要評估的 DataLoader (val_loader 或 test_loader)
    num_classes: 類別數 (預設 3: Pedestrian, Cyclist, Car)
    iou_thresholds:
        - float: 所有類別用同一個 IoU 閾值
        - list/tuple 長度 = num_classes: 每個類別一個 IoU 閾值
        - dict: key=class_id, value=iou_threshold
          例如: {0:0.25, 1:0.25, 2:0.5}
    device: torch.device
    class_names: list[str]，長度 = num_classes，用於輸出時顯示
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 轉成 per-class IoU threshold list
    per_class_iou_thr = _normalize_iou_thresholds(iou_thresholds, num_classes)

    model = model.to(device)
    model.eval()

    # 每個 class 記錄：所有 prediction 的 score / tp / fp
    all_scores = {c: [] for c in range(num_classes)}
    all_tp = {c: [] for c in range(num_classes)}
    all_fp = {c: [] for c in range(num_classes)}
    # 每個 class 的 ground truth box 數量
    num_gt = {c: 0 for c in range(num_classes)}

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="[Eval] mAP", ncols=120)
        for data_dict in pbar:
            data_dict = _move_batch_to_device(data_dict, device)

            batched_pts = data_dict["batched_pts"]             # list[tensor]
            batched_gt_bboxes = data_dict["batched_gt_bboxes"] # list[tensor], (Ni, 7)
            batched_gt_labels = data_dict["batched_labels"]    # list[tensor], (Ni, )

            # 推論
            results = model(batched_pts=batched_pts, mode="test")

            batch_size = len(batched_pts)
            for b_idx in range(batch_size):
                # 取出該 frame 的 GT
                gt_boxes = batched_gt_bboxes[b_idx].detach().cpu().numpy()  # (Ng, 7)
                gt_labels = batched_gt_labels[b_idx].detach().cpu().numpy() # (Ng, )

                # 取出該 frame 的 predict
                pred_dict = results[b_idx]
                pred_boxes = pred_dict["lidar_bboxes"].astype(np.float32)   # (Np, 7)
                pred_labels = pred_dict["labels"].astype(np.int64)          # (Np, )
                pred_scores = pred_dict["scores"].astype(np.float32)        # (Np, )

                # 逐 class 計算 TP/FP
                for cls_id in range(num_classes):
                    # 該類別使用的 IoU 閾值
                    iou_thr_cls = per_class_iou_thr[cls_id]

                    # 該 frame 的 GT 中屬於 cls_id 的
                    gt_mask = (gt_labels == cls_id)
                    gt_boxes_c = gt_boxes[gt_mask]
                    n_gt_c = gt_boxes_c.shape[0]
                    num_gt[cls_id] += n_gt_c

                    # 該 frame 的 prediction 中屬於 cls_id 的
                    pred_mask = (pred_labels == cls_id)
                    pred_boxes_c = pred_boxes[pred_mask]
                    pred_scores_c = pred_scores[pred_mask]

                    if pred_boxes_c.shape[0] == 0:
                        continue

                    # 按 score 由高到低排序
                    order = np.argsort(-pred_scores_c)
                    pred_boxes_c = pred_boxes_c[order]
                    pred_scores_c = pred_scores_c[order]

                    # 準備 TP/FP 判定
                    tp = np.zeros(pred_boxes_c.shape[0], dtype=np.float32)
                    fp = np.zeros(pred_boxes_c.shape[0], dtype=np.float32)

                    if n_gt_c == 0:
                        # 該類別在這個 frame 沒有 GT，所有都是 FP
                        fp[:] = 1.0
                    else:
                        # 計算 IoU (BEV)
                        pred_bev = boxes_to_bev(pred_boxes_c)
                        gt_bev = boxes_to_bev(gt_boxes_c)
                        ious = bev_iou(pred_bev, gt_bev)   # (Np_c, Ng_c)

                        gt_used = np.zeros(n_gt_c, dtype=bool)

                        for i_det in range(pred_boxes_c.shape[0]):
                            iou_row = ious[i_det]
                            max_iou = np.max(iou_row)
                            max_idx = np.argmax(iou_row)

                            if max_iou >= iou_thr_cls and not gt_used[max_idx]:
                                tp[i_det] = 1.0
                                gt_used[max_idx] = True
                            else:
                                fp[i_det] = 1.0

                    all_scores[cls_id].extend(list(pred_scores_c))
                    all_tp[cls_id].extend(list(tp))
                    all_fp[cls_id].extend(list(fp))

    # 統整成 AP / mAP
    ap_per_class = {}
    for cls_id in range(num_classes):
        scores = np.asarray(all_scores[cls_id], dtype=np.float32)
        tp = np.asarray(all_tp[cls_id], dtype=np.float32)
        fp = np.asarray(all_fp[cls_id], dtype=np.float32)
        npos = num_gt[cls_id]

        if npos == 0:
            ap = 0.0
        else:
            if scores.size == 0:
                ap = 0.0
            else:
                # 依 score 由高到低排序
                order = np.argsort(-scores)
                tp = tp[order]
                fp = fp[order]

                tp_cum = np.cumsum(tp)
                fp_cum = np.cumsum(fp)

                recall = tp_cum / float(npos)
                precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)

                ap = compute_ap(recall, precision)

        if class_names is not None and 0 <= cls_id < len(class_names):
            name = class_names[cls_id]
        else:
            name = f"class_{cls_id}"

        ap_per_class[name] = ap

    mAP = float(np.mean(list(ap_per_class.values()))) if len(ap_per_class) > 0 else 0.0

    return {
        "AP_per_class": ap_per_class,
        "mAP": mAP,
        "num_gt": num_gt,
        "iou_thresholds": per_class_iou_thr,
    }


###############################################
# 使用範例
###############################################
if __name__ == "__main__":
    # 假設你已經有 val_dataloader，或用 get_vod_train_val_dataloaders 取出
    # 這裡示範用你前面定義的 get_vod_train_val_dataloaders

    # from torch.utils.data import DataLoader
    # 這裡直接引用你前面定義好的:
    # - get_vod_train_val_dataloaders
    # - PointPillars
    # 若在同一個檔案中，直接使用即可；若拆成多檔案請自行 import.

    # calib_dir = 'view_of_delft_PUBLIC/radar/training/calib'
    # image_dir = 'view_of_delft_PUBLIC/lidar/training/image_2'
    # label_dir = 'view_of_delft_PUBLIC/lidar/training/label_2'
    # velodyne_dir = 'view_of_delft_PUBLIC/radar/training/velodyne'

    # _, val_loader, _, val_dataset = get_vod_train_val_dataloaders(
    #     calib_dir=calib_dir,
    #     image_dir=image_dir,
    #     label_dir=label_dir,
    #     velodyne_dir=velodyne_dir,
    #     batch_size=8,
    #     num_workers=0,
    #     val_ratio=0.2,
    #     seed=42,
    #     used_classes=['Car', 'Pedestrian', 'Cyclist'],  # 依你使用的類別順序調整
    # )

    # 類別順序要跟 training label id 一致
    class_names = ['Pedestrian', 'Cyclist', 'Car']  # 對應 CLASSES 的 id: 0,1,2

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 建 model 並載入你訓練好的 checkpoint
    num_classes = 3
    # 這裡假設你有 PointPillars 類別可用
    # from your_module import PointPillars, get_vod_train_val_dataloaders
    model = PointPillars(nclasses=num_classes).to(device)

    ckpt_path = "./runs_pointpillars/checkpoints/best_model.pth"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"{ckpt_path} 不存在，請確認路徑或先訓練並存 best_model.pth")

    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)

    # 這裡示範 Car 0.5，其餘類別 0.25：
    # class_names: [Pedestrian, Cyclist, Car] -> [0.25, 0.25, 0.5]
    per_class_iou = [0.25, 0.25, 0.5]

    eval_res = evaluate_pointpillars_map(
        model=model,
        dataloader=val_loader,
        num_classes=num_classes,
        iou_thresholds=per_class_iou,
        device=device,
        class_names=class_names,
    )

    print("========== mAP 結果 ==========")
    for cls_name, ap in eval_res["AP_per_class"].items():
        print(f"{cls_name}: AP = {ap:.4f}")
    print(f"mAP = {eval_res['mAP']:.4f}")
    print("GT 數量:", eval_res["num_gt"])
    print("各類別 IoU 閾值:", eval_res["iou_thresholds"])


[Eval] mAP: 100%|█████████████████████████████████████████████████████████████████████| 315/315 [00:58<00:00,  5.35it/s]

========== mAP 結果 ==========
Pedestrian: AP = 0.7285
Cyclist: AP = 0.8690
Car: AP = 0.9012
mAP = 0.8329
GT 數量: {0: 3963, 1: 1595, 2: 3931}
各類別 IoU 閾值: [0.25, 0.25, 0.5]


In [6]:
next(iter(val_loader))

{'batched_pts': [tensor([[ 1.0444e+00,  1.0793e+01, -1.1298e+00,  1.0934e+02],
          [ 6.9446e-01,  1.0713e+01, -1.0749e+00,  1.6391e+02],
          [-1.2268e+00,  3.3769e+01,  4.3031e-01,  2.5500e+02],
          ...,
          [-7.0453e-02,  1.0499e+01, -1.4758e+00,  1.5434e+02],
          [ 6.9569e-01,  1.0415e+01, -2.1272e+00,  1.3985e+02],
          [ 1.5627e-01,  1.0399e+01, -2.0121e+00,  1.5236e+02]]),
  tensor([[  0.4391,   5.2814,  -0.4415, 135.9916],
          [  0.2991,   5.3104,  -0.4233, 119.5976],
          [ -0.2954,   5.5221,   0.2430,  96.8325],
          ...,
          [  0.5551,   5.2532,  -0.7125, 106.4240],
          [  0.3019,   5.3143,  -0.6752, 115.8189],
          [  0.4165,   5.2778,  -0.9475,  92.0738]]),
  tensor([[ 1.1508e+00,  1.5453e+01, -1.6474e+00,  2.5500e+02],
          [-1.5551e+00,  5.4585e+01,  5.6826e-01,  2.4857e+02],
          [ 5.9630e-01,  1.8330e+01, -1.8516e+00,  1.3543e+02],
          ...,
          [-2.4099e-01,  1.1604e+01, -1.6457e+00

In [9]:
import cv2
import numpy as np
import torch
import os
import matplotlib.pyplot as plt

# 定義類別顏色 (BGR 格式)
COLORS = {
    0: (0, 255, 255),   # Pedestrian: 黃色
    1: (255, 255, 0),   # Cyclist: 青色
    2: (0, 255, 0),     # Car: 綠色
}

def get_lidar_3d_corners(boxes):
    """
    計算 3D Bounding Box 的 8 個頂點座標 (在 LiDAR 座標系下)
    boxes: (N, 7) [x, y, z, w, l, h, ry]
    return: (N, 8, 3)
    """
    if boxes.shape[0] == 0:
        return np.zeros((0, 8, 3))

    # boxes: x, y, z, w, l, h, ry
    # LiDAR 座標系定義: x 前, y 左, z 上
    # 這裡假設 boxes 格式符合 PointPillars 常見輸出
    
    dx = boxes[:, 3] / 2.0  # w
    dy = boxes[:, 4] / 2.0  # l
    dz = boxes[:, 5] / 2.0  # h
    
    # 建立 8 個頂點的相對座標 (x, y, z)
    # 順序通常為：底面 4 點 -> 頂面 4 點
    # 0: (+, +, -), 1: (+, -, -), 2: (-, -, -), 3: (-, +, -)
    # 4: (+, +, +), 5: (+, -, +), 6: (-, -, +), 7: (-, +, +)
    x_corners = np.array([1, 1, -1, -1, 1, 1, -1, -1], dtype=np.float32)
    y_corners = np.array([1, -1, -1, 1, 1, -1, -1, 1], dtype=np.float32)
    z_corners = np.array([-1, -1, -1, -1, 1, 1, 1, 1], dtype=np.float32)
    
    # (N, 8)
    x_corners = x_corners[None, :] * dx[:, None]
    y_corners = y_corners[None, :] * dy[:, None]
    z_corners = z_corners[None, :] * dz[:, None]
    
    # 旋轉 (Yaw / ry) - 繞 Z 軸旋轉
    # Rotation matrix around Z axis
    ry = boxes[:, 6]
    c, s = np.cos(ry), np.sin(ry)
    
    # x_rot = x * cos - y * sin
    # y_rot = x * sin + y * cos
    x_rot = x_corners * c[:, None] - y_corners * s[:, None]
    y_rot = x_corners * s[:, None] + y_corners * c[:, None]
    z_rot = z_corners
    
    # 平移 (加上中心點座標)
    x_final = x_rot + boxes[:, 0:1]
    y_final = y_rot + boxes[:, 1:2]
    z_final = z_rot + boxes[:, 2:3]
    
    # Stack 成 (N, 8, 3)
    corners_3d = np.stack([x_final, y_final, z_final], axis=-1)
    return corners_3d

def project_to_image(corners_3d, calib_info):
    """
    將 LiDAR 3D 點投影到 2D 圖像平面
    corners_3d: (N, 8, 3)
    calib_info: dict (containing P2, R0_rect, Tr_velo_to_cam)
    return: (N, 8, 2)  [u, v]
    """
    if len(corners_3d) == 0:
        return []

    # 取出參數矩陣
    P2 = calib_info['P2']             # (3, 4)
    R0 = calib_info['R0_rect']        # (3, 3) or (4, 4)
    Tr = calib_info['Tr_velo_to_cam'] # (3, 4) or (4, 4)
    
    # 確保矩陣維度一致 (都轉成 4x4)
    def to_4x4(mat):
        if mat.shape == (3, 3):
            tmp = np.eye(4)
            tmp[:3, :3] = mat
            return tmp
        elif mat.shape == (3, 4):
            tmp = np.eye(4)
            tmp[:3, :4] = mat
            return tmp
        return mat

    R0_4x4 = to_4x4(R0)
    Tr_4x4 = to_4x4(Tr)
    
    # 組合投影矩陣: Lidar -> Camera -> Image
    # World(Lidar) -> Camera = R0 * Tr
    lidar_to_cam = R0_4x4 @ Tr_4x4
    
    N, num_points, _ = corners_3d.shape
    pts_3d = corners_3d.reshape(-1, 3) # (N*8, 3)
    
    # 變成齊次座標 (x, y, z, 1)
    pts_3d_hom = np.hstack((pts_3d, np.ones((pts_3d.shape[0], 1)))) # (N*8, 4)
    
    # 1. 轉到 Rectified Camera Coords (X_c, Y_c, Z_c)
    pts_cam_hom = pts_3d_hom @ lidar_to_cam.T # (N*8, 4)
    
    # 2. 剔除 Z <= 0 的點 (在相機後面的點)
    # 這裡為了繪圖簡單，先保留，但繪製時若有頂點在後面可能會顯示異常
    
    # 3. 投影到 Image Plane (u, v, z)
    # P2 是 3x4，所以只取 pts_cam_hom 的前 4 個維度運算 (其實本來就是4)
    # 這裡 P2 乘的是 Camera 座標
    pts_img_hom = pts_cam_hom @ P2.T # (N*8, 3)
    
    # 4. Normalize: u = x/z, v = y/z
    pts_img_hom[:, 0] /= pts_img_hom[:, 2]
    pts_img_hom[:, 1] /= pts_img_hom[:, 2]
    
    uv = pts_img_hom[:, :2].reshape(N, 8, 2)
    return uv

def draw_3d_boxes_on_image(image, corners_2d, labels, scores=None, thickness=2):
    """
    在圖片上畫出 3D 線框
    image: cv2 image (BGR)
    corners_2d: (N, 8, 2)
    labels: (N, )
    """
    img_draw = image.copy()
    
    # 定義 12 條連線 (對應 get_lidar_3d_corners 的頂點順序)
    # 底面: 0-1, 1-2, 2-3, 3-0
    # 頂面: 4-5, 5-6, 6-7, 7-4
    # 支柱: 0-4, 1-5, 2-6, 3-7
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7)
    ]
    
    for i in range(len(corners_2d)):
        box = corners_2d[i] # (8, 2)
        label = int(labels[i])
        color = COLORS.get(label, (255, 255, 255))
        
        # 畫線
        for s, e in edges:
            pt1 = tuple(box[s].astype(int))
            pt2 = tuple(box[e].astype(int))
            cv2.line(img_draw, pt1, pt2, color, thickness, cv2.LINE_AA)
        
        # 畫前方十字 (0, 1, 5, 4) 是前面板 (假設 x 前, y 左, 根據 corners 定義需確認)
        # 根據上面的定義：
        # 0:(+,+,-), 1:(+,-,-), 5:(+,-,+), 4:(+,+,+) -> 這些都是 x 正向 (車頭)
        front_face = [0, 1, 5, 4]
        p0 = box[0].astype(int)
        p5 = box[5].astype(int)
        p1 = box[1].astype(int)
        p4 = box[4].astype(int)
        cv2.line(img_draw, tuple(p0), tuple(p5), color, 1, cv2.LINE_AA)
        cv2.line(img_draw, tuple(p1), tuple(p4), color, 1, cv2.LINE_AA)

        # 畫 Label 和 Score
        if scores is not None:
            text = f"{label} {scores[i]:.2f}"
            cv2.putText(img_draw, text, tuple(box[4].astype(int)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    return img_draw

def visualize_one_batch(batch_dict, pred_dicts, out_dir="vis_result"):
    """
    主呼叫函數：處理一個 Batch 的視覺化
    batch_dict: DataLoader 出來的資料 (含 image_path, calib)
    pred_dicts: Model 推論出來的結果 (list of dict)
    """
    os.makedirs(out_dir, exist_ok=True)
    
    batch_size = len(batch_dict['batched_pts'])
    
    for b in range(batch_size):
        # 1. 讀取圖片
        img_info = batch_dict['batched_img_info'][b]
        img_path = img_info['image_path']
        
        # 處理路徑問題 (Windows/Linux 分隔符)
        if '\\' in img_path:
            img_path = img_path.replace('\\', '/')
            
        if not os.path.exists(img_path):
            print(f"Warning: Image not found {img_path}")
            continue
            
        image = cv2.imread(img_path)
        
        # 2. 取得預測結果 (過濾低分)
        preds = pred_dicts[b]
        pred_boxes = preds['lidar_bboxes'].cpu().numpy() # (N, 7)
        pred_labels = preds['labels'].cpu().numpy()      # (N, )
        pred_scores = preds['scores'].cpu().numpy()      # (N, )
        
        mask = pred_scores > 0.3 # 視覺化閾值
        pred_boxes = pred_boxes[mask]
        pred_labels = pred_labels[mask]
        pred_scores = pred_scores[mask]
        
        # 3. 取得 GT (若想要同時畫 GT 比較)
        gt_boxes = batch_dict['batched_gt_bboxes'][b].cpu().numpy()
        gt_labels = batch_dict['batched_labels'][b].cpu().numpy()
        
        # 4. 計算 3D 頂點並投影
        calib = batch_dict['batched_calib_info'][b]
        
        # --- 處理 Prediction ---
        corners_3d_pred = get_lidar_3d_corners(pred_boxes)
        corners_2d_pred = project_to_image(corners_3d_pred, calib)
        
        # 畫 Prediction (實線)
        img_vis = draw_3d_boxes_on_image(image, corners_2d_pred, pred_labels, pred_scores, thickness=2)
        
        # --- 處理 Ground Truth (選用) ---
        # corners_3d_gt = get_lidar_3d_corners(gt_boxes)
        # corners_2d_gt = project_to_image(corners_3d_gt, calib)
        # # GT 用紅色畫，或者稍微細一點
        # img_vis = draw_3d_boxes_on_image(img_vis, corners_2d_gt, gt_labels, thickness=1)

        # 5. 儲存
        filename = os.path.basename(img_path)
        save_path = os.path.join(out_dir, f"vis_{filename}")
        cv2.imwrite(save_path, img_vis)
        print(f"Saved: {save_path}")

# run_vis.py (範例)

import torch
import numpy as np
# 假設你的模型定義在 model.py
# from model import PointPillars 

def main():
    # 1. 設定路徑與載入資料
    calib_dir = 'view_of_delft_PUBLIC/lidar/training/calib'
    image_dir = 'view_of_delft_PUBLIC/lidar/training/image_2'
    label_dir = 'view_of_delft_PUBLIC/lidar/training/label_2'
    velodyne_dir = 'view_of_delft_PUBLIC/lidar/training/velodyne'



    
    with torch.no_grad():
        for i, data_dict in enumerate(val_loader):
            if i > 5: break # 只畫前 5 個 Batch
            
            data_dict = _move_batch_to_device(data_dict, device)
            
            # --- 這裡替換成真正的 model inference ---
            # results = model(batched_pts=data_dict['batched_pts'], mode="test")
            
            # 這裡我為了演示，直接把 Ground Truth 當作 Prediction 傳進去畫出來
            # 這樣你可以先測試繪圖功能是否正常
            results = []
            batch_size = len(data_dict['batched_pts'])
            for b in range(batch_size):
                fake_pred = {
                    'lidar_bboxes': data_dict['batched_gt_bboxes'][b], # 借用 GT 當預測
                    'labels': data_dict['batched_labels'][b],
                    'scores': torch.ones_like(data_dict['batched_labels'][b], dtype=torch.float32)
                }
                results.append(fake_pred)
            # --------------------------------------

            print(f"Visualizing batch {i}...")
            visualize_one_batch(data_dict, results, out_dir="visualization_output")

if __name__ == "__main__":
    main()

Visualizing batch 0...
Saved: visualization_output\vis_06912.jpg
Saved: visualization_output\vis_04456.jpg
Saved: visualization_output\vis_09669.jpg
Saved: visualization_output\vis_07512.jpg
Visualizing batch 1...
Saved: visualization_output\vis_01577.jpg
Saved: visualization_output\vis_04134.jpg
Saved: visualization_output\vis_00296.jpg
Saved: visualization_output\vis_05015.jpg
Visualizing batch 2...
Saved: visualization_output\vis_04705.jpg
Saved: visualization_output\vis_09643.jpg
Saved: visualization_output\vis_04248.jpg
Saved: visualization_output\vis_00635.jpg
Visualizing batch 3...
Saved: visualization_output\vis_04738.jpg
Saved: visualization_output\vis_04899.jpg
Saved: visualization_output\vis_01213.jpg
Saved: visualization_output\vis_01080.jpg
Visualizing batch 4...
Saved: visualization_output\vis_01583.jpg
Saved: visualization_output\vis_00707.jpg
Saved: visualization_output\vis_00756.jpg
Saved: visualization_output\vis_04208.jpg
Visualizing batch 5...
Saved: visualization_o